In [ ]:
import gym
import sys
from gym.envs.registration import register
sys.path.append('./Base2048Env')
import Base2048Env
register(
    id='2048-v0',
    entry_point='Base2048Env:Base2048Env'
)

In [42]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

class PPOAgent:
    def __init__(self, state_dim, action_dim, lr=1e-4, gamma=0.99, clip_epsilon=0.2, update_epochs=4):
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.gamma = gamma
        self.clip_epsilon = clip_epsilon
        self.update_epochs = update_epochs
        
        # Policy network
        self.policy_net = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, action_dim),
            nn.Softmax(dim=-1)
        )
        
        # Value network
        self.value_net = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )
        
        self.optimizer = optim.Adam(
            list(self.policy_net.parameters()) + list(self.value_net.parameters()), lr=lr
        )
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.policy_net.to(self.device)
        self.value_net.to(self.device)

    def select_action(self, state_tensor):
        state_tensor = state_tensor.to(self.device)
        probs = self.policy_net(state_tensor)
        dist = torch.distributions.Categorical(probs)
        action = dist.sample()
        log_prob = dist.log_prob(action)
        value = self.value_net(state_tensor).squeeze(-1)
        return action.cpu().numpy(), log_prob, value

    def compute_advantages_and_returns(self, rewards, dones, values, next_value):
        advantages = []
        returns = []
        gae = torch.zeros_like(next_value)  # Initialize GAE with the same shape as next_value

        for step in reversed(range(len(rewards))):
            delta = rewards[step] + (1 - dones[step]) * self.gamma * next_value - values[step]
            gae = delta + self.gamma * (1 - dones[step]) * gae
            advantages.insert(0, gae)
            returns.insert(0, gae + values[step])
            next_value = values[step]

        advantages = torch.stack(advantages)
        returns = torch.stack(returns)
        return advantages.to(self.device), returns.to(self.device)

    def optimize(self, states, actions, log_probs, advantages, returns):
        for _ in range(self.update_epochs):
            # Compute current log probabilities and entropy
            probs = self.policy_net(states)  # Shape [batch_size * timesteps, action_dim]
            dist = torch.distributions.Categorical(probs)
            
            # Ensure actions are flattened correctly
            actions = actions.view(-1)  # Flatten actions to match the batch size * timesteps
            new_log_probs = dist.log_prob(actions)  # Compute log probabilities of the actions
            entropy = dist.entropy().mean()

            # Compute policy loss with clipping
            ratio = (new_log_probs - log_probs.view(-1)).exp()  # Flatten log_probs for batch comparison
            clipped_ratio = torch.clamp(ratio, 1 - self.clip_epsilon, 1 + self.clip_epsilon)
            print(ratio)
            print(advantages.view(-1))
            surr1 = ratio * advantages.view(-1)


            surr2 = clipped_ratio * advantages.view(-1)
            policy_loss = -torch.min(surr1, surr2).mean()

            # Compute value loss (Mean Squared Error)
            values = self.value_net(states).squeeze(-1)  # Shape [batch_size * timesteps]
            value_loss = nn.functional.mse_loss(values, returns.view(-1))

            # Total loss
            loss = policy_loss + 0.5 * value_loss - 0.01 * entropy

            # Update policy and value networks
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()

def run_game(agent, env, batch_size):
    states, actions, log_probs, rewards, dones, values = [], [], [], [], [], []
    state = np.array([env.reset().flatten() for _ in range(batch_size)])  # Updated initialization
    done = np.zeros(batch_size, dtype=bool)
    
    while not done.all():
        state_tensor = torch.tensor(state, dtype=torch.float32).to(agent.device)
        action, log_prob, value = agent.select_action(state_tensor)
        next_state, reward, done, _ = zip(*[env.step(a) for a in action])
        # print(next_state)
        done = np.array(done, dtype=bool)
        
        # Process and store results
        next_state = np.array([ns.flatten() for ns in next_state])
        states.append(state_tensor)
        actions.append(torch.tensor(action).to(agent.device))
        log_probs.append(log_prob)
        rewards.append(torch.tensor(reward, dtype=torch.float32).to(agent.device))
        dones.append(torch.tensor(done, dtype=torch.float32).to(agent.device))
        values.append(value)
        
        state = next_state

    max_tile = np.max(state)
    next_state_tensor = torch.tensor(state, dtype=torch.float32).to(agent.device)
    next_value = agent.value_net(next_state_tensor).squeeze(-1)

    print("Steps: {}\nMax tile: {}".format(len(rewards), max_tile))

    return states, actions, log_probs, rewards, dones, values, next_value

def train_agent(agent, env, batch_size=20, max_episodes=1000):
    for episode in range(max_episodes):
        # Gather batch data
        states, actions, log_probs, rewards, dones, values, next_value = run_game(agent, env, batch_size)
        
        # Compute advantages and returns
        advantages, returns = agent.compute_advantages_and_returns(
            rewards=torch.cat(rewards),
            dones=torch.cat(dones),
            values=torch.cat(values),
            next_value=next_value
        )
        
        # Optimize policy and value networks
        agent.optimize(
            states=torch.cat(states),
            actions=torch.cat(actions),
            log_probs=torch.cat(log_probs),
            advantages=advantages,
            returns=returns
        )

        print(f"Episode {episode + 1}:")


agent = PPOAgent(state_dim=16, action_dim=4)
env = gym.make("2048-v0")
train_agent(agent, env, max_episodes=1000)

c:\Users\Huaichen\Desktop\gym\Base2048Env.py:125: FutureWarning: Using a non-tuple sequence for multidimensional indexing is deprecated; use `arr[tuple(seq)]` instead of `arr[seq]`. In the future this will be interpreted as an array index, `arr[np.array(seq)]`, which will result either in an error or a different result.
  board[tile_locs] = tiles
c:\Python310\lib\site-packages\gym\utils\passive_env_checker.py:31: UserWarning: WARN: A Box observation space has an unconventional shape (neither an image, nor a 1D vector). We recommend flattening the observation to have only a 1D vector or use a custom policy to properly process the data. Actual observation shape: (4, 4)
  logger.warn(
c:\Python310\lib\site-packages\gym\utils\passive_env_checker.py:174: UserWarning: WARN: Future gym versions will require that `Env.reset` can be passed a `seed` instead of using `Env.seed` for resetting the environment random number generator.
  logger.warn(
c:\Python310\lib\site-packages\gym\utils\passive_e

Steps: 6
Max tile: 64
tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.], device='cuda:0',
       grad_fn=<ExpBackward0>)
tensor([413.6779, 413.6779, 413.6779,  ...,  -1.3634,  -1.3634,  -1.3634],
       device='cuda:0', grad_fn=<ViewBackward0>)


RuntimeError: The size of tensor a (120) must match the size of tensor b (2400) at non-singleton dimension 0